# Prepare a simulated data


In [51]:
import pandas as pd

dt_agg = pd.DataFrame({
    "sex": ["F", "M", "F", "M"],
    "drug": ["N", "N", "Y", "Y"],
    "ha": [1, 12, 3, 7],
    "no_ha": [21, 30, 40, 13]
})

In [58]:
dt_ind = (
    dt_agg.melt(
        id_vars=["sex", "drug"],
        value_vars=["ha", "no_ha"],
        var_name="ha_status",
        value_name="count"
    )
    .assign(ha=lambda x: (x["ha_status"] == "ha").astype(int))
    .loc[lambda x: x.index.repeat(x["count"])]
    .drop(columns=["ha_status", "count"])
    .reset_index(drop=True)
)

dt_ind.to_csv("ggb.csv", index=False)

In [53]:
dt_ind.groupby(["drug", "sex", "ha"]).size().unstack(fill_value=0)

ha         0   1
drug sex        
N    F    21   1
     M    30  12
Y    F    40   3
     M    13   7

In [54]:
rel_freq = pd.crosstab(
    [dt_ind["drug"], dt_ind["sex"]],
    dt_ind["ha"],
    normalize="index"
).rename(columns={0: "no_ha", 1: "ha"})

100 * rel_freq

ha            no_ha         ha
drug sex                      
N    F    95.454545   4.545455
     M    71.428571  28.571429
Y    F    93.023256   6.976744
     M    65.000000  35.000000

In [55]:
import statsmodels.formula.api as smf

lm = smf.ols("ha ~ drug * sex", data=dt_ind).fit()
print(lm.summary())

                            OLS Regression Results                            
Dep. Variable:                     ha   R-squared:                       0.104
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     4.784
Date:                Fri, 27 Mar 2026   Prob (F-statistic):            0.00346
Time:                        13:19:35   Log-Likelihood:                -52.008
No. Observations:                 127   AIC:                             112.0
Df Residuals:                     123   BIC:                             123.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.0455      0

In [57]:
smf.ols("ha ~ drug", data=dt_ind ).fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     ha   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.005
Method:                 Least Squares   F-statistic:                    0.4167
Date:                Fri, 27 Mar 2026   Prob (F-statistic):              0.520
Time:                        13:20:02   Log-Likelihood:                -58.805
No. Observations:                 127   AIC:                             121.6
Df Residuals:                     125   BIC:                             127.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2031      0.048      4.193      0.000       0.107       0.299
drug[T.Y]     -0.0444      0.069     -0.645      0.520      -0.181       0.092
==============================================================================
Omnibus:                       37.869   Durbin-Watson:                   0.058
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               60.358
Skew:                           1.648   Prob(JB):                     7.83e-14
Kurtosis:                       3.738   Cond. No.                         2.61
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""